# Attention et transformeurs : activités en classe
**MATH60629 — Apprentissage automatique I — Semaine 10**

Ce carnet accompagne les diapositives de la séance. Il contient quatre activités :

1. **Le jeu du goulot d'étranglement** (≈ 10 min) : pourquoi un encodeur-décodeur RNN a du mal avec les longues séquences.
2. **Self-attention à la main** (≈ 15 min, en équipes de deux) : calculer requêtes, clés, valeurs et sorties sur un petit exemple, puis le masque causal.
3. **Embeddings de position** (≈ 10 min) : ce que le self-attention « voit » quand on change l'ordre des mots.
4. **Attention multi-têtes** (≈ 15 min) : plusieurs têtes, plusieurs notions de similarité ; exploration d'un vrai modèle (CamemBERT).

Exécutez les cellules dans l'ordre. Les cellules marquées **Solution** sont masquées : double-cliquez pour voir le code, ou exécutez-les pour voir le résultat.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def softmax(S, axis=-1):
    S = S - S.max(axis=axis, keepdims=True)  # stabilité numérique
    E = np.exp(S)
    return E / E.sum(axis=axis, keepdims=True)

---
## 1. Le jeu du goulot d'étranglement

Dans un encodeur-décodeur RNN, **toute** la phrase source doit passer par un seul vecteur de taille fixe (l'état caché final $h_T$) avant que le décodeur puisse produire la traduction. On va jouer le rôle de ce vecteur.

**Règles du jeu**

1. Formez des équipes de deux : l'**encodeur** (A) et le **décodeur** (B).
2. Le décodeur détourne les yeux. L'encodeur lit le paragraphe projeté (cellule suivante) pendant une minute.
3. L'encodeur résume le paragraphe en **exactement 5 mots** et les transmet au décodeur (rien d'autre : pas de gestes, pas de ponctuation créative !).
4. Le décodeur reconstruit le paragraphe le plus fidèlement possible, en deux minutes.
5. On compare. Quelles informations ont survécu ? Lesquelles ont disparu en premier ?

*Variante :* refaites l'exercice avec 15 mots. Puis avec 5 mots, mais en permettant au décodeur de poser une question ciblée à l'encodeur **à chaque phrase qu'il écrit**. C'est exactement ce que fait l'attention.

In [ ]:
#@title Paragraphe à projeter (encodeurs seulement)
paragraphe = (
    "Jean-Baptiste Poquelin, dit Molière, naît à Paris en 1622 dans une famille de tapissiers. "
    "Plutôt que de reprendre la charge de son père, il fonde en 1643 l'Illustre Théâtre avec la "
    "comédienne Madeleine Béjart. La troupe fait faillite, et Molière passe ensuite près de treize "
    "ans à jouer en province. De retour à Paris en 1658, il obtient la protection de Louis XIV et "
    "écrit certaines des comédies les plus jouées du répertoire français, dont Tartuffe, "
    "Le Misanthrope et L'Avare. Il s'effondre sur scène en 1673 pendant une représentation du "
    "Malade imaginaire et meurt quelques heures plus tard."
)
print(paragraphe)

In [ ]:
# Encodeur (A) : exactement 5 mots
resume = ["", "", "", "", ""]

# Décodeur (B) : votre reconstruction
reconstruction = '''
'''

In [ ]:
#@title Comparer la reconstruction à l'original
import re

def mots_pleins(texte):
    # Mots de plus de 3 lettres, en minuscules : une mesure grossière du contenu
    return {m for m in re.findall(r"[\wÀ-ÿ'-]+", texte.lower()) if len(m) > 3}

assert len([m for m in resume if m.strip()]) <= 5, "Le résumé doit contenir au plus 5 mots !"
original = mots_pleins(paragraphe)
retrouves = original & mots_pleins(reconstruction)
print(f"Mots pleins retrouvés : {len(retrouves)} / {len(original)}")
print("Retrouvés :", sorted(retrouves))
print("Perdus    :", sorted(original - retrouves))

### Le même phénomène, en chiffres

Une expérience simple : chaque mot d'un vocabulaire de 5 000 mots reçoit un vecteur aléatoire de dimension $d$. On « encode » une séquence de $n$ mots en **additionnant** leurs vecteurs (un vecteur de taille fixe, comme $h_T$). Puis on essaie de retrouver quels mots étaient dans la séquence en cherchant les $n$ vecteurs les plus similaires au résumé.

Cet encodeur est très simplifié (il ignore même l'ordre des mots, ce qui rend la tâche *plus facile* !), mais il illustre la contrainte de capacité.

**Avant d'exécuter :** comment le taux de rappel va-t-il évoluer quand $n$ augmente ? Et quand $d$ augmente ?

In [ ]:
def rappel_moyen(d, n, V=5000, essais=200, graine=0):
    rng = np.random.default_rng(graine)
    E = rng.normal(size=(V, d)) / np.sqrt(d)   # un vecteur par mot du vocabulaire
    total = 0.0
    for _ in range(essais):
        mots = rng.choice(V, size=n, replace=False)
        h = E[mots].sum(axis=0)                 # le « goulot » : un seul vecteur de taille d
        scores = E @ h
        devines = np.argpartition(-scores, n)[:n]
        total += len(set(devines) & set(mots)) / n
    return total / essais

longueurs = [1, 2, 5, 10, 20, 50, 100]
plt.figure(figsize=(7, 4))
for d in [16, 64, 256]:
    plt.plot(longueurs, [rappel_moyen(d, n) for n in longueurs], marker="o", label=f"d = {d}")
plt.xscale("log")
plt.xlabel("Longueur de la séquence n")
plt.ylabel("Proportion de mots retrouvés")
plt.title("Un vecteur de taille fixe ne peut pas tout retenir")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**À discuter**

* Pour une taille $d$ donnée, à partir de quelle longueur la reconstruction se dégrade-t-elle ?
* Augmenter $d$ aide, mais à quel coût ? (Pensez au nombre de paramètres d'un RNN, qui croît en $d^2$.)
* Avec l'attention, le décodeur a accès à **tous** les $h_j$ : la représentation grandit avec la séquence. Que deviendrait cette courbe ?

---
## 2. Self-attention à la main (en équipes de deux)

Trois mots, chacun représenté par un vecteur de dimension $D = 2$ :

$$
X = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \end{pmatrix}
  = \begin{pmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{pmatrix}
\qquad \text{(« le », « chat », « dort »)}
$$

Les matrices de projection :

$$
W_q = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \qquad
W_k = \begin{pmatrix} 1 & 0 \\ 1 & 1 \end{pmatrix}, \qquad
W_v = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}
$$

**Convention.** La ligne $i$ correspond au mot qui « regarde » (sa **requête** $q_i = x_i W_q$), la colonne $j$ au mot regardé (sa **clé** $k_j = x_j W_k$). Le score est $e_{ij} = q_i \cdot k_j$, les poids sont $a_{ij} = \mathrm{softmax}_j(e_{ij})$ et la sortie du mot $i$ est $z_i = \sum_j a_{ij}\, v_j$, avec $v_j = x_j W_v$. (C'est la convention de Vaswani et al. Pour simplifier, on omet ici la division par $\sqrt{D}$.)

**Sur papier :**

1. Calculez $Q = XW_q$, $K = XW_k$ et $V = XW_v$.
2. Calculez la matrice des scores $E = QK^\top$ (3 × 3).
3. Appliquez le softmax à chaque **ligne**. Table utile : $e^0 = 1$, $e^1 \approx 2{,}72$, $e^2 \approx 7{,}39$, $e^3 \approx 20{,}09$.
4. Calculez les sorties $z_1, z_2, z_3$.
5. Quel mot « dort » regarde-t-il le plus ? Est-ce raisonnable, vu la façon dont on a construit $X$ ?

In [ ]:
#@title Solution : vérifiez vos calculs { display-mode: "form" }
X  = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
Wq = np.array([[1, 0], [0, 1]], dtype=float)
Wk = np.array([[1, 0], [1, 1]], dtype=float)
Wv = np.array([[0, 1], [1, 0]], dtype=float)
mots = ["le", "chat", "dort"]

Q, K, V = X @ Wq, X @ Wk, X @ Wv
E = Q @ K.T
A = softmax(E, axis=1)
Z = A @ V

print("Q =\n", Q, "\nK =\n", K, "\nV =\n", V)
print("\nScores E = QKᵀ =\n", E)
print("\nPoids d'attention A (chaque ligne somme à 1) =\n", A)
print("\nSorties Z = AV =\n", Z)

### Et si on multipliait les scores par 10 ?

**Prédisez d'abord :** à quoi ressemblera la matrice d'attention si tous les scores sont multipliés par 10 ? Par 0,1 ?

Indice : dans un vrai transformeur, $D$ vaut souvent 64 ou 128 par tête, et un produit scalaire entre deux vecteurs de dimension $D$ a une variance qui croît avec $D$.

In [ ]:
#@title Solution { display-mode: "form" }
for facteur in [0.1, 1, 10]:
    print(f"Scores × {facteur} :\n", softmax(facteur * E, axis=1), "\n")

print("Avec ×10, le softmax devient presque un « argmax » : chaque mot ne regarde presque plus que")
print("le ou les mots de score maximal, et le gradient du softmax devient presque nul. D'où la division par √D dans")
print("Vaswani et al. : elle garde les scores dans une plage où l'apprentissage reste stable.")

### Le masque causal (décodeur)

Dans un décodeur, le mot $i$ ne peut porter attention qu'aux mots $j \le i$ (il ne peut pas « voir l'avenir »).

**Sur papier :** dans la matrice des scores $E$ ci-dessus, barrez les cases interdites. Recalculez ensuite le softmax de chaque ligne en ne gardant que les cases permises. Que vaut $a_{11}$ ? Pourquoi ?

Avec cette convention (lignes = requêtes), quelle forme a la matrice d'attention : triangulaire supérieure ou inférieure ?

In [ ]:
#@title Solution { display-mode: "form" }
masque = np.triu(np.ones((3, 3), dtype=bool), k=1)   # True au-dessus de la diagonale = interdit
E_masque = np.where(masque, -np.inf, E)
A_causal = softmax(E_masque, axis=1)
print("Scores masqués :\n", E_masque)
print("\nAttention causale :\n", A_causal)
print("\nLe premier mot ne peut regarder que lui-même : a₁₁ = 1.")
print("Lignes = requêtes, colonnes = clés : la matrice est triangulaire INFÉRIEURE.")
print("(Elle serait triangulaire supérieure avec la convention transposée.)")

---
## 3. Embeddings de position

On prend deux phrases qui contiennent **exactement les mêmes mots**, dans un ordre différent :

* « le chat mange la plante »
* « la plante mange le chat »

On fait passer chacune dans une couche de self-attention (poids aléatoires, $D = 8$), **sans** embeddings de position.

**Avant d'exécuter :** la représentation de sortie de « chat » sera-t-elle la même dans les deux phrases ? Et la représentation moyenne de la phrase ?

In [ ]:
rng = np.random.default_rng(42)
D = 8
vocabulaire = ["le", "chat", "mange", "la", "plante"]
embedding = {mot: rng.normal(size=D) for mot in vocabulaire}
Wq3, Wk3, Wv3 = (rng.normal(size=(D, D)) / np.sqrt(D) for _ in range(3))

def self_attention(X):
    Q, K, V = X @ Wq3, X @ Wk3, X @ Wv3
    A = softmax(Q @ K.T / np.sqrt(X.shape[1]), axis=1)
    return A @ V

phrase1 = "le chat mange la plante".split()
phrase2 = "la plante mange le chat".split()

def comparer(Z1, Z2, titre):
    print(titre)
    for mot in vocabulaire:
        z1 = Z1[phrase1.index(mot)]
        z2 = Z2[phrase2.index(mot)]
        print(f"  {mot:>7} : écart entre les deux phrases = {np.abs(z1 - z2).max():.2e}")
    print(f"  Représentation moyenne de la phrase : écart = {np.abs(Z1.mean(0) - Z2.mean(0)).max():.2e}\n")

X1 = np.stack([embedding[m] for m in phrase1])
X2 = np.stack([embedding[m] for m in phrase2])
comparer(self_attention(X1), self_attention(X2), "Sans embeddings de position :")

Les écarts sont nuls (à l'erreur d'arrondi près) : sans information de position, le self-attention traite une phrase comme un **ensemble** de mots. Changer l'ordre des mots en entrée change seulement l'ordre des sorties ; on dit que le self-attention est *équivariant par permutation*. Pour lui, « le chat mange la plante » et « la plante mange le chat » sont la même chose.

**Remède :** on ajoute à chaque mot un vecteur qui dépend de sa position. Ici, on utilise les embeddings sinusoïdaux de Vaswani et al. (dans beaucoup de modèles, ils sont plutôt appris).

In [ ]:
def embeddings_position(n, D):
    pos = np.arange(n)[:, None]
    i = np.arange(0, D, 2)[None, :]
    angles = pos / (10000 ** (i / D))
    P = np.zeros((n, D))
    P[:, 0::2] = np.sin(angles)
    P[:, 1::2] = np.cos(angles)
    return P

P = embeddings_position(5, D)
comparer(self_attention(X1 + P), self_attention(X2 + P), "Avec embeddings de position :")

plt.figure(figsize=(6, 2.5))
plt.imshow(embeddings_position(50, 16).T, aspect="auto", cmap="RdBu")
plt.xlabel("Position dans la séquence")
plt.ylabel("Dimension")
plt.title("Embeddings de position sinusoïdaux (D = 16)")
plt.colorbar()
plt.show()

**À discuter**

* Pourquoi **additionner** le vecteur de position plutôt que le concaténer ? Quels sont les avantages et inconvénients de chaque option ?
* Que se passe-t-il si une phrase en test est plus longue que toutes les phrases vues à l'entraînement ? (Comparez les embeddings sinusoïdaux et appris.)
* Un RNN a-t-il besoin d'embeddings de position ? Pourquoi ?

---
## 4. Attention multi-têtes

### 4a. Deux têtes sur notre petit exemple

Chaque tête a ses propres matrices $W_q^h, W_k^h, W_v^h$ et peut donc capturer une notion différente de similarité. Les sorties des têtes sont concaténées puis projetées :
$x'_j = [z_j^1 \; z_j^2 \; \cdots \; z_j^H]\, W_0$.

On reprend les trois mots de la section 2 avec deux têtes.

In [ ]:
tetes = {
    "tête 1 (celle de la section 2)": (Wq, Wk, Wv),
    "tête 2": (np.array([[0, 1], [1, 0]], float),
               np.array([[1, 0], [0, 1]], float),
               np.array([[1, 0], [0, 1]], float)),
}

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
sorties = []
for ax, (nom, (Wq_h, Wk_h, Wv_h)) in zip(axes, tetes.items()):
    A_h = softmax((X @ Wq_h) @ (X @ Wk_h).T, axis=1)
    sorties.append(A_h @ (X @ Wv_h))
    ax.imshow(A_h, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(mots)
    ax.set_yticks(range(3)); ax.set_yticklabels(mots)
    ax.set_xlabel("regarde (clé)"); ax.set_ylabel("mot (requête)")
    ax.set_title(nom)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{A_h[i, j]:.2f}", ha="center", va="center")
plt.tight_layout()
plt.show()

concat = np.concatenate(sorties, axis=1)               # 3 × (D'·H) = 3 × 4
W0 = np.random.default_rng(1).normal(size=(4, 2))      # (D'·H) × D
print("Concaténation des têtes :\n", concat)
print("\nSortie du MHSA, x' = [z¹ z²] W0 :\n", concat @ W0)

**À discuter :** les deux têtes regardent-elles les mêmes mots ? Pourquoi est-ce utile d'avoir plusieurs têtes plutôt qu'une seule, plus grande ?

### 4b. Explorer les têtes d'un vrai modèle : CamemBERT

CamemBERT est un transformeur **encodeur** (de type BERT) entraîné sur du texte français : 12 couches × 12 têtes = 144 têtes d'attention.

On regarde une phrase de type « schéma de Winograd », où un pronom ne peut être résolu qu'avec le sens de la phrase :

* « L'animal n'a pas traversé la rue parce qu'**il** était trop fatigué. » (il = l'animal)
* « L'animal n'a pas traversé la rue parce qu'**elle** était trop large. » (elle = la rue)

**Avant d'explorer :** à votre avis, quels types de relations différentes têtes pourraient-elles apprendre ? (syntaxe, mot suivant, ponctuation, coréférence…)

La première cellule installe BertViz et télécharge le modèle (environ 450 Mo, une minute ou deux).

In [ ]:
!pip -q install bertviz

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from bertviz import head_view

nom_modele = "almanach/camembert-base"
try:
    tokenizer = AutoTokenizer.from_pretrained(nom_modele)
    modele = AutoModel.from_pretrained(nom_modele, attn_implementation="eager")
except Exception:
    nom_modele = "camembert-base"
    tokenizer = AutoTokenizer.from_pretrained(nom_modele)
    modele = AutoModel.from_pretrained(nom_modele, attn_implementation="eager")
modele.eval()

def attentions(phrase):
    enc = tokenizer(phrase, return_tensors="pt")
    with torch.no_grad():
        sortie = modele(**enc, output_attentions=True)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    return sortie.attentions, tokens   # tuple de 12 tenseurs (1, têtes, n, n)

phrase_il = "L'animal n'a pas traversé la rue parce qu'il était trop fatigué."
phrase_elle = "L'animal n'a pas traversé la rue parce qu'elle était trop large."

att_il, tokens_il = attentions(phrase_il)
print("Tokens :", tokens_il)

Explorez librement : choisissez une couche dans le menu, cliquez sur les couleurs pour isoler des têtes, et survolez un mot pour voir à quoi il porte attention. Cherchez une tête où « il » regarde « animal ».

Remarquez que beaucoup de têtes envoient l'essentiel de leur attention vers le premier token `<s>` ou vers la ponctuation : c'est un comportement fréquent, souvent interprété comme une façon pour la tête de « ne rien faire ».

In [ ]:
head_view(att_il, tokens_il)

### Chercher systématiquement

Plutôt que de chercher à l'œil parmi 144 têtes, on peut calculer, pour chaque tête, l'attention que le pronom porte à « animal » et à « rue ».

In [ ]:
def indices(tokens, mot):
    return [i for i, t in enumerate(tokens) if t.replace("▁", "").lower() == mot]

def attention_vers(att, tokens, source, cible):
    i_src, i_cib = indices(tokens, source), indices(tokens, cible)
    if not i_src or not i_cib:
        raise ValueError(f"Introuvable : {source!r} ou {cible!r} dans {tokens}")
    A = torch.stack([a[0] for a in att])      # (couches, têtes, n, n)
    return A[:, :, i_src][:, :, :, i_cib].sum(-1).mean(-1).numpy()   # (couches, têtes)

att_elle, tokens_elle = attentions(phrase_elle)
il_animal  = attention_vers(att_il,   tokens_il,   "il",   "animal")
il_rue     = attention_vers(att_il,   tokens_il,   "il",   "rue")
elle_animal = attention_vers(att_elle, tokens_elle, "elle", "animal")
elle_rue    = attention_vers(att_elle, tokens_elle, "elle", "rue")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, M, titre in [(axes[0], il_animal, "« il » → « animal »"),
                     (axes[1], elle_rue, "« elle » → « rue »")]:
    im = ax.imshow(M, cmap="viridis")
    ax.set_xlabel("Tête"); ax.set_ylabel("Couche"); ax.set_title(titre)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Les 5 têtes où « il » regarde le plus « animal » :")
for c, t in np.dstack(np.unravel_index(np.argsort(-il_animal, axis=None)[:5], il_animal.shape))[0]:
    print(f"  couche {c + 1:>2}, tête {t + 1:>2} : il→animal = {il_animal[c, t]:.2f}, il→rue = {il_rue[c, t]:.2f}"
          f"  |  elle→animal = {elle_animal[c, t]:.2f}, elle→rue = {elle_rue[c, t]:.2f}")

**À discuter**

* Les têtes qui relient « il » à « animal » relient-elles aussi « elle » à « rue » ? Si oui, que pourrait faire cette tête ? Sinon, qu'est-ce qui pourrait l'expliquer ? (Pensez au genre grammatical : en français, « il » et « animal » sont masculins, « elle » et « rue » sont féminins. Est-ce que la tête résout vraiment la coréférence, ou suit-elle simplement l'accord ?)
* Proposez une paire de phrases qui permettrait de distinguer ces deux hypothèses, et testez-la.
* Une attention élevée veut-elle dire que le modèle « utilise » cette information pour sa prédiction ? (L'attention montre où l'information circule, pas nécessairement ce qui compte pour la sortie.)